# First layer of the Self-attention mechanism
## Input

## Libraries

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import pandas as pd
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

## Colors for graphics

In [ ]:
# ── Setting up the configuration ───────────────────────────────────────────────
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# ── Colors ───────────────────────────────────────────────────────────────────
AZUL_OSCURO   = "#1F3A5F"
AZUL_CLARO    = "#5DADE2"
VERDE_OSCURO  = "#1E8449"
VERDE_CLARO   = "#58D68D"
ROJO_OSCURO   = "#922B21"
ROJO_CLARO    = "#EC7063"
NARANJA       = "#E67E22"
AMARILLO      = "#F4D03F"
MORADO        = "#7D3C98"
GRIS_OSCURO   = "#566573"
DARK_BG       = '#0F0F1A'
PANEL_BG      = '#1A1A2E'
ACCENT1       = '#E94560'
ACCENT2       = '#1a5276'
ACCENT3       = '#16213E'
TEXT_CLR      = '#E0E0E0'
GRID_CLR      = '#2A2A45'
NORMAL_CLR    = '#FF1500'
GOLD          = '#FFD700'
CYAN          = '#00D4FF'
BLANCO        = '#FFFFFF'

## Graphic visualization for the embedding

In [ ]:
np.set_printoptions(precision = 3, suppress = True)

# -------------------------------------------------------
# 1. Tokenization
# -------------------------------------------------------
text = "All data in deep learning must be represented as vectors."
tokens = text.lower().replace(".", " .").split()

vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)))}

seq_len = len(tokens)
d_model = 3

# -------------------------------------------------------
# 2. Simulated Token Embeddings
# -------------------------------------------------------
rng = np.random.default_rng(seed = 42)

embedding_table = rng.normal(
    0,
    1,
    size = (len(vocab), d_model)
)

token_ids = np.array([vocab[tok] for tok in tokens])
token_embeddings = embedding_table[token_ids]

# -------------------------------------------------------
# 3. Positional Encoding
# -------------------------------------------------------
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i   = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (i // 2)) / d_model
    )

    angles = pos * angle_rates
    pe     = np.zeros((seq_len, d_model))

    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])

    return pe


pos_encodings = positional_encoding(seq_len, d_model)

# -------------------------------------------------------
# 4. Final encoder input
# -------------------------------------------------------
final_input = token_embeddings + pos_encodings

# -------------------------------------------------------
# 5. Interactive 3D Plot (Professional Version)
# -------------------------------------------------------

fig = go.Figure()
colors = plt.cm.tab20(np.linspace(0, 1, len(tokens)))

for token, vec, color in zip(tokens, final_input, colors):
    rgb = tuple((np.array(color[:3]) * 255).astype(int))
    color_rgb = f"rgb({rgb[0]},{rgb[1]},{rgb[2]})"
    x, y, z = vec

    # ------------------------------------------
    # Shaft of the vector
    # ------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x = [0, x],
            y = [0, y],
            z = [0, z],
            mode = "lines",
            line = dict(
                color = color_rgb,
                width = 6
            ),
            hoverinfo  = "skip",
            showlegend = False
        )
    )

    # ------------------------------------------
    # Arrow head (Cone)
    # ------------------------------------------
    direction = vec / np.linalg.norm(vec)

    fig.add_trace(
        go.Cone(
            x = [x],
            y = [y],
            z = [z],

            u = [direction[0]],
            v = [direction[1]],
            w = [direction[2]],

            anchor = "tip",

            sizemode = "absolute",
            sizeref  = 0.25,

            colorscale = [
                [0, color_rgb],
                [1, color_rgb]
            ],

            showscale = False,
            hoverinfo = "skip"
        )
    )

    # ------------------------------------------
    # Endpoint + Label
    # ------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x = [x],
            y = [y],
            z = [z],
            mode   = "markers+text",
            marker = dict(
                size  = 5,
                color = color_rgb
            ),
            text = [token],
            textposition = "top center",
            textfont = dict(
                family = "Arial",
                size   = 13,
                color  = color_rgb
            ),
            hovertemplate =
            "<b>%{text}</b><br>" +
            "x = %{x:.3f}<br>"   +
            "y = %{y:.3f}<br>"   +
            "z = %{z:.3f}<extra></extra>",
            showlegend = False
        )
    )

# -------------------------------------------------------
# Axis limits
# -------------------------------------------------------

lim = np.abs(final_input).max() * 1.35

# -------------------------------------------------------
# Layout
# -------------------------------------------------------

fig.update_layout(
    title = dict(
        text = "<b>Encoder Input Vectors</b><br>Token Embedding + Positional Encoding",
        x    = 0.5,
        font = dict(
            size = 22,
            color = "white"
        )
    ),
    paper_bgcolor = "rgb(8,10,18)",
    plot_bgcolor  = "rgb(8,10,18)",

    margin = dict(
        l = 0,
        r = 0,
        b = 0,
        t = 70
    ),
    scene = dict(
        bgcolor    = "rgb(8,10,18)",
        aspectmode = "cube",
        camera = dict(
            eye = dict(
                x = 1.55,
                y = 1.55,
                z = 1.25
            )
        ),

        xaxis = dict(
            title = "Dimension 0",
            range = [-lim, lim],
            backgroundcolor = "rgb(8,10,18)",
            gridcolor       = "rgb(80,80,80)",
            zerolinecolor   = "white",
            showbackground  = True,
            color           = "white"
        ),

        yaxis = dict(
            title = "Dimension 1",
            range = [-lim, lim],
            backgroundcolor = "rgb(8,10,18)",
            gridcolor       = "rgb(80,80,80)",
            zerolinecolor   = "white",
            showbackground  = True,
            color           = "white"
        ),

        zaxis = dict(
            title = "Dimension 2",
            range = [-lim, lim],
            backgroundcolor = "rgb(8,10,18)",
            gridcolor       = "rgb(80,80,80)",
            zerolinecolor   = "white",
            showbackground  = True,
            color           = "white"
        )
    )
)

fig.show()

For all instances and purposes, it's a 3D embedding; basically, it’s the easiest way to represent how the vectors aim in different ways depending on the context
